# llm_budget

> Pricing, the token profile, cost functions and the spending cap for the Claude API runs (Phase 10 and the follow-ups): `PRICES` with the date they were checked, `TOKEN_PROFILE` measured in the Phase 10 pilot, `costOf`, `estimateCost`, `maxCallCost`, and `Budget`, a cap shared by processes through a locked ledger.

In [ ]:
#| default_exp llm_budget

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import fcntl
import json
from pathlib import Path

PRICES = {   # $ per million tokens (Haiku: prompts up to 100K tokens), from the pricing page 2026-10-12 (pilot, T7)
    'claude-haiku-5-5': dict(input=0.10, output=0.50, cacheWrite=0.125, cacheRead=0.01, checked='2026-10-12'),
    'claude-sonnet-5-5': dict(input=2.00, output=10.00, cacheWrite=2.50, cacheRead=0.10, checked='2026-10-12'),
}
TOKEN_PROFILE = {   # per call: (uncached input, cache-read input, output) tokens, measured in the pilot (T7, 2026-10-12)
    ('claude-haiku-5-5', 'forage'): (840, 1824, 433), ('claude-haiku-5-5', 'board'): (208, 1771, 901),
    ('claude-sonnet-5-5', 'forage'): (840, 1824, 433), ('claude-sonnet-5-5', 'board'): (214, 1743, 805),   # forage: unused
    ('claude-haiku-5-5', 'clue'): (208, 1771, 901), ('claude-sonnet-5-5', 'clue'): (214, 1743, 805),   # clue: provisional (board's)
}
USAGE = ('input_tokens', 'cache_creation_input_tokens', 'cache_read_input_tokens', 'output_tokens')


class BudgetExceeded(RuntimeError):
    pass


def costOf(usage, price):
    "Dollars for one reply's usage."
    return (usage.get('input_tokens', 0) * price['input'] + usage.get('cache_creation_input_tokens', 0) * price['cacheWrite']
            + usage.get('cache_read_input_tokens', 0) * price['cacheRead'] + usage.get('output_tokens', 0) * price['output']) / 1e6


def estimateCost(model, task, agents, turns, prices=PRICES, profile=TOKEN_PROFILE):
    "Expected dollars for `agents` x `turns` calls from the token profile."
    i, r, o = profile[(model, task)]
    p = prices[model]
    return agents * turns * (i * p['input'] + r * p['cacheRead'] + o * p['output']) / 1e6



def maxCallCost(model, task, maxTokens, prices=PRICES, profile=TOKEN_PROFILE):
    "Dollars for one call that uses its whole output allowance (uncached and cache-read input from the profile)."
    i, r, _ = profile[(model, task)]
    p = prices[model]
    return (i * p['input'] + r * p['cacheRead'] + maxTokens * p['output']) / 1e6

class Budget:
    "A spending cap shared by processes through an append-only ledger under a file lock."
    def __init__(self, cap, ledger):
        self.cap, self.ledger = float(cap), Path(ledger)

    def spent(self):
        if not self.ledger.exists():
            return 0.0
        with open(self.ledger) as f:
            fcntl.flock(f, fcntl.LOCK_SH)
            total = sum(json.loads(line)['cost'] for line in f if line.strip())
            fcntl.flock(f, fcntl.LOCK_UN)
        return total

    def remaining(self):
        return self.cap - self.spent()

    def check(self, need=0.0):
        left = self.remaining()
        if left <= 0 or need > left:
            raise BudgetExceeded(f'need ${need:.4f}, ${left:.4f} left of the ${self.cap:.2f} cap')

    def add(self, cost, note=''):
        self.ledger.parent.mkdir(parents=True, exist_ok=True)
        with open(self.ledger, 'a') as f:
            fcntl.flock(f, fcntl.LOCK_EX)
            f.write(json.dumps(dict(cost=float(cost), note=note)) + '\n')
            fcntl.flock(f, fcntl.LOCK_UN)

## Tests

In [ ]:
import tempfile
from pathlib import Path
from explRL.llm_budget import *

# a call that uses its whole output allowance: profile input plus maxTokens output
i, r, _ = TOKEN_PROFILE[('claude-sonnet-5-5', 'board')]
assert abs(maxCallCost('claude-sonnet-5-5', 'board', 8000) - (i * 2.0 + r * 0.10 + 8000 * 10.0) / 1e6) < 1e-15
assert maxCallCost('claude-sonnet-5-5', 'board', 8000) < 6 / 64                # 64 such calls in flight cost under $6
assert maxCallCost('claude-haiku-5-5', 'forage', 2000) < maxCallCost('claude-haiku-5-5', 'forage', 8000)
assert costOf(dict(output_tokens=10**6), PRICES['claude-haiku-5-5']) == 0.5
# the budget: a shared ledger, a cap, and a check for what a run still needs
tmp = Path(tempfile.mkdtemp())
b = Budget(1.0, tmp / 'spend.jsonl')
b.add(0.25, 'x'); b.add(0.25, 'y')
assert b.spent() == 0.5 and b.remaining() == 0.5 and Budget(1.0, tmp / 'spend.jsonl').spent() == 0.5
b.check(0.4)
try:
    b.check(0.6); raise AssertionError
except BudgetExceeded:
    pass

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()